# Silver: customers

A typical Silver notebook using the shared `fabric_utils` library from the custom Environment.
The notebook only holds the business logic for this table; paths, reading, writing, schema, logging and shared business rules (`add_surrogate_key`) come from the library, and Spark config comes from the Environment.

In [ ]:
# Parameters (can be overridden by a pipeline or runMultiple)
load_from = None   # e.g. "2026-01-01" for an incremental load, None for full

In [ ]:
from fabric_utils import *

paths = get_paths()
log = NotebookLogger("nb_silver_customers", layer="silver")

destination_schema = StructType([
    StructField("sk_customer", LongType(), False),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("created_at", DateType(), True),
    StructField("modified_at", TimestampType(), True),
    StructField("date_load", TimestampType(), True),
])

In [ ]:
try:
    df = load_data(paths["path_bronze"], "customers",
                   filter_column="ModifiedAt" if load_from else None, filter_from=load_from)

    df = rename_columns(df, {
        "CustomerId": "customer_id",
        "Name": "customer_name",
        "Country": "country",
        "CreatedAt": "created_at",
        "ModifiedAt": "modified_at",
    })
    df = trim_string_columns(df)
    df = deduplicate(df, keys=["customer_id"], order_by="modified_at")
    df = add_surrogate_key(df, keys=["customer_id"], name="sk_customer")   # enterprise rule, same in every notebook
    df = add_date_load(df, tz="Europe/Madrid")
    df = enforce_destination_schema(df, destination_schema)

    save_table(df, paths["path_silver"], "customers")
    log.end()
except Exception as e:
    log.log_error(e)
    raise

### Exceptions stay local

If one step needs a config the rest of the session shouldn't get, scope it instead of calling `spark.conf.set()`:

```python
# A legacy parquet source written with the old calendar
with session_config(spark, spark__sql__parquet__datetimeRebaseModeInRead="LEGACY"):
    df_legacy = spark.read.parquet(paths["path_bronze_files"] + "legacy/customers/")

# One query that regressed on NEE: run just that block on the JVM engine
with native_disabled(spark):
    save_table(df_problematic, paths["path_silver"], "customers_history")
```